In [2]:
import os
import sys
import time
import pickle
import pandas as pd
import numpy as np
from datetime import datetime

# =============================================================================
# SHARED CONFIGURATION PARAMETERS
# =============================================================================

# -- File & Path Configuration --
HDF_FILE_PATH = '../../data/raw/all_hourly_data.h5'
FEATURE_CLASSIFICATION_PATH = '../../data/processed/eda_results_corrected/feature_classification.csv'
OUTPUT_DIR = 'phase_1_outputs'

# -- Experiment Configuration --
DRY_RUN = True  # Set to False for full dataset
DRY_RUN_PATIENTS = 2000  # Number of patients for dry run
USE_CACHED_PREPROCESSING = True  # Use cache if available

# -- Feature Engineering Options --
CALCULATE_TRENDS = True  # Include trend calculations
WINDOW_SIZE = 24  # Hours of data for feature extraction
GAP_TIME = 6      # Hours of gap before prediction

# -- Target and Reproducibility --
TARGET_VARIABLE = 'mort_hosp'
SEED = 42

# -- Hyperparameter Tuning Configuration --
N_OPTUNA_TRIALS = 15  # Number of trials for hyperparameter search
OPTUNA_TIMEOUT = 1800 # Timeout in seconds

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [3]:
print("=" * 70)
print("PHASE 1: CLEAN DATA PREPROCESSING")
print("=" * 70)

preprocessing_start = time.time()

# Import and run preprocessing
try:
    # Import the clean preprocessing script as a module
    import importlib.util
    spec = importlib.util.spec_from_file_location("data_preprocessing_clean", "data_preprocessing_clean.py")
    if spec is None:
        raise ImportError("Could not load data_preprocessing_clean.py")
    
    preprocessing_module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError("No loader found for data_preprocessing_clean.py")
    
    # Execute the preprocessing script
    print(f"Starting clean data preprocessing at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    spec.loader.exec_module(preprocessing_module)
    
    # Create configuration dictionary from notebook parameters
    config = {
        'HDF_FILE_PATH': HDF_FILE_PATH,
        'FEATURE_CLASSIFICATION_PATH': FEATURE_CLASSIFICATION_PATH,
        'OUTPUT_DIR': OUTPUT_DIR,
        'DRY_RUN': DRY_RUN,
        'DRY_RUN_PATIENTS': DRY_RUN_PATIENTS,
        'USE_CACHED_PREPROCESSING': USE_CACHED_PREPROCESSING,
        'CALCULATE_TRENDS': CALCULATE_TRENDS,
        'WINDOW_SIZE': WINDOW_SIZE,
        'GAP_TIME': GAP_TIME,
        'TARGET_VARIABLE': TARGET_VARIABLE,
        'SEED': SEED
    }
    
    # Call the main function explicitly with configuration
    preprocessing_module.main(config)
    
    preprocessing_time = time.time() - preprocessing_start
    print(f"\n✓ Clean data preprocessing completed in {preprocessing_time/60:.2f} minutes")
    
except Exception as e:
    print(f"❌ Error during preprocessing: {e}")
    raise


PHASE 1: CLEAN DATA PREPROCESSING
Starting clean data preprocessing at 2025-07-01 22:44:45


2025-07-01 22:44:45,817 [INFO] - Using cached data - skipping preprocessing!



✓ Clean data preprocessing completed in 0.01 minutes


In [5]:
print("=" * 70)
print("PHASE 2: XGBOOST MODEL TRAINING & EVALUATION")
print("=" * 70)

xgboost_start = time.time()

# Initialize variables for results capture
auroc_results = None
auprc_results = None
final_model = None
xgboost_results = None

try:
    # Import and run XGBoost analysis script  
    import importlib.util
    spec = importlib.util.spec_from_file_location("xgboost_analysis", "xgboost_analysis.py")
    if spec is None:
        raise ImportError("Could not load xgboost_analysis.py")
    
    xgboost_module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError("No loader found for xgboost_analysis.py")
    
    print(f"Starting XGBoost analysis at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Execute the XGBoost analysis module and call main function
    spec.loader.exec_module(xgboost_module)
    
    # Create configuration dictionary from notebook parameters
    config = {
        'OUTPUT_DIR': OUTPUT_DIR,
        'DRY_RUN': DRY_RUN,
        'DRY_RUN_PATIENTS': DRY_RUN_PATIENTS,
        'CALCULATE_TRENDS': CALCULATE_TRENDS,
        'WINDOW_SIZE': WINDOW_SIZE,
        'GAP_TIME': GAP_TIME,
        'TARGET_VARIABLE': TARGET_VARIABLE,
        'SEED': SEED,
        'N_OPTUNA_TRIALS': N_OPTUNA_TRIALS,
        'OPTUNA_TIMEOUT': OPTUNA_TIMEOUT
    }
    
    # Call the main function explicitly with configuration
    xgboost_module.main(config)
    
    # Load the results that were saved by the XGBoost script
    results_path = os.path.join(OUTPUT_DIR, 'results_xgboost_baseline.pkl')
    if os.path.exists(results_path):
        with open(results_path, 'rb') as f:
            xgboost_results = pickle.load(f)
        
        # Extract key metrics for display
        auroc_results = {
            'point_estimate': xgboost_results['test_auroc'],
            'ci_lower': xgboost_results['test_auroc_ci_lower'],
            'ci_upper': xgboost_results['test_auroc_ci_upper']
        }
        
        auprc_results = {
            'point_estimate': xgboost_results['test_auprc'], 
            'ci_lower': xgboost_results['test_auprc_ci_lower'],
            'ci_upper': xgboost_results['test_auprc_ci_upper']
        }
        
        print(f"\n✅ XGBoost Results Successfully Captured:")
        print(f"   AUROC: {auroc_results['point_estimate']:.4f} "
              f"(95% CI: {auroc_results['ci_lower']:.4f}-{auroc_results['ci_upper']:.4f})")
        print(f"   AUPRC: {auprc_results['point_estimate']:.4f} "
              f"(95% CI: {auprc_results['ci_lower']:.4f}-{auprc_results['ci_upper']:.4f})")
    else:
        print("⚠️  Results file not found - metrics may not be available in summary")
    
    xgboost_time = time.time() - xgboost_start
    print(f"\n✓ XGBoost analysis completed in {xgboost_time/60:.2f} minutes")
    
except Exception as e:
    print(f"❌ Error during XGBoost analysis: {e}")
    print(f"   This may be due to XGBoost library installation issues")
    print(f"   Preprocessing completed successfully - data is ready for manual analysis")
    xgboost_time = time.time() - xgboost_start
    print(f"   Attempted runtime: {xgboost_time/60:.2f} minutes")
    # Don't raise - allow pipeline to continue and show preprocessing results


2025-07-01 22:51:57,660 [INFO] - Loading preprocessed data with prefix: preprocessed_mort_hosp_dryrun_2000_trends_True_window_24_gap_6_seed_42
2025-07-01 22:51:57,666 [INFO] - Data shapes: X_train=(847, 392), X_val=(122, 392), X_test=(324, 392)
2025-07-01 22:51:57,666 [INFO] - Performing data validation for XGBoost...


PHASE 2: XGBOOST MODEL TRAINING & EVALUATION
Starting XGBoost analysis at 2025-07-01 22:51:57
❌ Error during XGBoost analysis: Cannot interpret '<attribute 'dtype' of 'numpy.generic' objects>' as a data type
   This may be due to XGBoost library installation issues
   Preprocessing completed successfully - data is ready for manual analysis
   Attempted runtime: 0.00 minutes
